In [1]:
import pandas as pd
from Bio import Medline
from Bio import Entrez
import string 

Entrez.email = "1219131962@qq.com"
Entrez.api_key = "6fcb01c2989c35aa1da3d9d0118abedfc409"
PMID_info = []
CGM_result_dict = {}

def get_keywords(PMID_list):
    """Thank to Dr. Mao Zhitao for his help in this function.
    """
    # if 'nan' in PMID_list: PMID_list.remove('nan')
    # PMID_list = [x for x in PMID_list if str(x) != 'nan']
    
    for i in PMID_list:
        # print(i)
        handle = Entrez.efetch(db="pubmed", id=i, rettype="medline", retmode="text")
        records = Medline.parse(handle)
        records = list(records)

        for record in records:
            title = record.get("TI","?")
            abstract = record.get("AB","?")
            keywords = record.get("OT", "?")
            Journal = record.get("JT", "?")
            country = record.get("AD", "?")[0].split(', ')[-1].strip(string.punctuation)
            pmid = record.get('PMID','?')
            Author = record.get('FAU','?')
            doi = record.get('SO','?')
            date = record.get('DEP','?')
            year = record.get("DP","?")
            institution = record.get('AD','?')[0]

            PMID_info.append({'PMID': pmid,
                            'DOI': doi,
                            'Author': Author,
                            'Title':title,
                            'Abstract': abstract,
                            'Keywords': keywords,
                            'Journal': Journal,
                            'Institution': institution,
                            'Country': country,
                            'Date': date,
                            'Year': year})
            
    for key, value in CGM_result_dict.items():
        PMID_info.append([key,value])

    return PMID_info

In [2]:
"""
测试用
"""
df = pd.read_csv('pmid-demo.txt', sep='\t', header=None, names=['PMID'])

# 删除重复值
df = df.dropna(subset=['PMID'])
df = df.reset_index(drop=True)

get_keywords(list(df.iloc[:, 0]))
df_info = pd.DataFrame(PMID_info)

df_info['DOI'] = df_info['DOI'].str.extract(r'(10\.\d{4,9}\/[-._;()/:A-Za-z0-9]+)')
df_info['DOI'] = df_info['DOI'].str.rstrip('.')

df_info

##### 解析出近10年来Mt在pubmed上的文献信息

In [9]:
"""
解析出近Mt,10年来在pubmed上的文献信息
"""
df_mt = pd.read_csv('pmid-Myceliopht-set.txt', sep='\t', header=None, names=['PMID'])

# 删除重复值
df_mt = df_mt.dropna(subset=['PMID'])
df_mt = df_mt.reset_index(drop=True)

# 拆分，每100个一组
df_mt1 = df_mt.iloc[0:100,:]
df_mt2 = df_mt.iloc[100:200,:]

In [ ]:
get_keywords(list(df_mt1.iloc[:, 0]))
df_mt1_paper = pd.DataFrame(PMID_info)

In [17]:
get_keywords(list(df_mt2.iloc[:, 0]))
df_mt2_paper = pd.DataFrame(PMID_info)

In [ ]:
# 合并
df_mt_paper = pd.concat([df_mt1_paper, df_mt2_paper], axis=0)
df_mt_paper = df_mt_paper.drop_duplicates(subset=['PMID'])
df_mt_paper = df_mt_paper.reset_index(drop=True)

# 保存
df_mt_paper.to_csv('Nc_paper.csv', sep=',', index=False)

# Keywords列中为？的行
mt_paper = df_mt_paper[df_mt_paper.Keywords != '?']
mt_paper.to_excel('Nc_Paper_filter.xlsx', index=False, sheet_name='Sheet1')

##### 解析出近10年来Nc在pubmed上的文献信息

In [ ]:
df_nc = pd.read_csv('pmid-Neurospora-set.txt', sep='\t', header=None, names=['PMID'])

# 删除重复值
df_nc = df_nc.dropna(subset=['PMID'])
df_nc = df_nc.reset_index(drop=True)

# 将df_nc拆分，每100个保存为一个dataframe
df_nc1 = df_nc.iloc[0:100,:]
df_nc2 = df_nc.iloc[100:200,:]
df_nc3 = df_nc.iloc[200:300,:]
df_nc4 = df_nc.iloc[300:400,:]
df_nc5 = df_nc.iloc[400:500,:]
df_nc6 = df_nc.iloc[500:600,:]
